# 📗 지식그래프 적재: 제약 먼저, 노드 그다음, 관계 마지막

지난 시간까지 우리는 **이미 만들어진 그래프** 위에서 세고, 요약하고, 빠르게 찾고, 규칙을 걸었습니다. 그 그래프를 넣어 준 것은 맨 위의 `[제공 코드]` 적재 셀이었습니다. 이번 시간에는 그 셀이 하던 일을 **여러분이 직접** 합니다.

새 문법은 거의 나오지 않습니다. 대신 **넣는 차례**가 전부입니다. 실무에서 데이터를 그래프에 넣을 때는 늘 같은 순서를 밟습니다. 오늘은 그 순서를 그대로 따라갑니다.

<img src="images/적재_순서_3단계.png" width="900">

오늘 밟을 순서입니다. 세 단계 모두 앞 시간에 배운 문법으로 합니다. 새로 배우는 것은 **어느 것을 언제 하느냐**입니다.

## ⏪ 복습: 지난 시간까지

- `count`·`sum`·`avg` 로 요약하고, `WITH` 로 집계한 값을 넘겨 **다시 집계**했습니다.
- **인덱스**는 찾아보기입니다. 레이블을 적어야 그 레이블의 인덱스를 씁니다.
- **UNIQUE 제약**은 문지기입니다. 규칙을 어기는 데이터를 아예 거부하고, **인덱스를 겸합니다.**
- `MERGE` 는 있으면 그대로, 없으면 만들기입니다. 여러 번 실행해도 결과가 같습니다(**멱등**).

제약과 인덱스를 이미 배웠으니, 오늘은 그것을 **실제 적재 절차 안에 제자리로 놓는** 일을 합니다.

**오늘의 목표**

**1. 왜 제약부터 설계할까요?**
- [ ] 자료의 어느 칸을 **키로 삼을지** 정하고, 그 선택이 그래프 모양을 바꾸는 것을 확인한다.
- [ ] 키에는 **`NODE KEY`** 가 맞는 이유를 알고, 적재보다 **먼저** 건다.

**2. 노드를 넣습니다**
- [ ] (2-1) `MERGE` 에는 **식별 키만** 넣고 나머지 값은 `SET` 으로 채운다(문자열·정수·실수).
- [ ] (2-1) 덮어쓰면 안 되는 **이력**은 `ON CREATE SET`·`ON MATCH SET` 으로 갈라 쓴다.
- [ ] (2-2) 한 줄씩 보내는 것과 `UNWIND` 로 **묶어 보내는 것**의 차이를 안다.

**3. 관계를 잇습니다**
- [ ] (3-1) 양 끝을 **같은 질의 안에서 `MATCH` 로 찾아** 잇는다.
- [ ] (3-2) 역과 역 **사이의 값**(거리·시간)을 관계의 속성으로 매단다.
- [ ] (3-3) 관계를 **세어** 환승역을 가려내고, 다시 돌려도 값이 같게 만든다.

오늘 만들 그래프의 모양입니다. 레이블 두 종류와 관계 두 종류뿐입니다.

| 넣을 것 | 모양 | 속성 |
|---|---|---|
| 역 | `(:Station {name})` | 이름. 적재 이력과 환승 여부가 붙습니다 |
| 호선 | `(:Line {name})` | 역수·총연장·기점·종점 |
| 운행 | `(:Line)-[:SERVES]->(:Station)` | 몇 번째 역인지, 기점부터의 누계 거리 |
| 인접 | `(:Station)-[:NEXT]->(:Station)` | 역간거리, 소요시간, 어느 호선인지 |

> 값이 **역에 붙는지, 역 사이에 붙는지**를 눈여겨보세요. 역간거리는 어느 한 역의 성질이 아니라 두 역 사이의 값이라 **관계**가 들고 있습니다.

<img src="images/지하철_그래프_조각.png" width="1000">

*시연에서 넣을 그래프입니다. 1호선 열 개 역과 2호선 여덟 개 역이 **시청에서 겹칩니다.** 노드에는 이름과 적재 기록이, 관계에는 거리·시간·정차 순서가 붙습니다.*

아래 준비 셀을 위에서부터 실행하세요. **연결 → 초기화** 두 개뿐입니다. 앞 시간과 달리 **적재 셀이 없습니다.** 오늘은 그 자리를 여러분이 채웁니다.

> ⚠️ **초기화 셀은 연결된 데이터베이스의 노드·관계·제약을 모두 지웁니다.** 오늘은 **적재 자체가 실습**이라 반드시 **빈 그래프**에서 시작해야 합니다. `.env` 가 실습 전용 DB 를 가리키는지 확인하고 실행하세요.

In [5]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

Neo4j 연결: bolt://localhost:7687


In [6]:
# [제공 코드] 실습 전용 DB 초기화: 이 데이터베이스를 통째로 비웁니다.
# ⚠️ 가리는 것 없이 **노드·관계·제약을 전부** 지웁니다.
run_cypher("MATCH (n) DETACH DELETE n")   # DETACH 는 노드에 붙은 관계까지 함께 지웁니다
# 제약조건은 노드를 지워도 남습니다. 이름을 조회해 하나씩 DROP 합니다
for _c in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name"):
    run_cypher("DROP CONSTRAINT " + _c["name"] + " IF EXISTS")
print("초기화 완료:", NEO4J_URI, "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

초기화 완료: bolt://localhost:7687 · 남은 노드: 0


아래 셀은 **넣을 값만** 파이썬으로 받아 둡니다. 그래프에는 아직 아무것도 넣지 않습니다.

시연은 **지하철**로 합니다. 1호선을 먼저 넣고 2호선을 이어 넣는데, 두 호선이 **시청 역에서 겹칩니다.** 그 겹침이 2절과 3절에서 그대로 드러납니다.

`🖐️ 함께 따라하기` 는 **의료 지식그래프**로 합니다. 같은 절차를 다른 데이터에 옮겨 써 보는 것이 목적이고, 무엇보다 **키가 갈립니다.** 지하철은 이름이 키인데 의료 자료는 `id` 가 키입니다. 왜 그런지는 1절에서 봅니다.

In [7]:
# [제공 코드] 오늘 적재할 조각: 이 셀은 실행만 하세요(값을 정의하기만 합니다).
# 서울교통공사 공개자료에서 1호선 열 개 역 전체와 2호선 앞 여덟 개 역을 떼어 왔습니다.

# --- 1호선: 서울역~청량리 ---
# [노드] :Line - 호선 하나. name 이 키이고 나머지는 그 노선을 설명하는 값입니다
#          name          : 호선 이름(문자열). 이 값이 키입니다
#          station_count : 노선 전체의 역 수(정수)
#          length_km     : 노선 전체의 총연장(실수, km)
#          first_station : 기점 역 이름(문자열)
#          last_station  : 종점 역 이름(문자열)
line1 = {"name": "1", "station_count": 10, "length_km": 7.8,
         "first_station": "서울역", "last_station": "청량리"}

# [노드] :Station - 역. 이름뿐입니다. 거리와 시간은 한 역의 성질이 아니라
#        역과 역 "사이" 의 값이라 노드가 아니라 관계가 들고 있습니다
stations1 = ["서울역", "시청", "종각", "종로3가", "종로5가",
             "동대문", "동묘앞", "신설동", "제기동", "청량리"]

# [관계] (:Line)-[:SERVES]->(:Station) - 그 호선이 이 역에 섭니다
#        자리는 (역 이름, stop_order, cumulative_km)
#          stop_order    : 그 호선에서 몇 번째로 서는 역인지(정수). 호선마다 1 부터 다시 셉니다
#                          원본 파일의 '연번' 은 전 노선 통번호라 값이 다릅니다
#          cumulative_km : 그 호선 기점에서 이 역까지의 누계 거리(실수, km)
serves1 = [
    ("서울역", 1, 0.0), ("시청", 2, 1.1), ("종각", 3, 2.1), ("종로3가", 4, 2.9),
    ("종로5가", 5, 3.8), ("동대문", 6, 4.6), ("동묘앞", 7, 5.2), ("신설동", 8, 5.9),
    ("제기동", 9, 6.8), ("청량리", 10, 7.8),
]

# [관계] (:Station)-[:NEXT]->(:Station) - 이 역 다음이 그 역입니다
#        자리는 (앞 역, 다음 역, distance_km, travel_time)
#          distance_km : 두 역 사이의 거리(실수, km)
#          travel_time : 두 역 사이를 가는 데 걸리는 시간(문자열, "분:초")
#        두 값 모두 어느 한 역의 성질이 아니라 구간의 값이라 관계에 붙습니다.
#        역이 열 개면 사이 구간은 아홉 개입니다
next1 = [
    ("서울역", "시청", 1.1, "02:00"), ("시청", "종각", 1.0, "02:00"),
    ("종각", "종로3가", 0.8, "01:30"), ("종로3가", "종로5가", 0.9, "01:30"),
    ("종로5가", "동대문", 0.8, "01:30"), ("동대문", "동묘앞", 0.6, "01:00"),
    ("동묘앞", "신설동", 0.7, "01:30"), ("신설동", "제기동", 0.9, "01:30"),
    ("제기동", "청량리", 1.0, "01:30"),
]

# --- 2호선: 시청~왕십리 ---
# 네 값의 구조는 1호선과 똑같습니다. 다른 점만 셋 적어 둡니다.
#   1) 역수 52·총연장 60.2km 는 **노선 전체**의 값입니다. 오늘 넣는 것은 앞 여덟 개 역뿐입니다
#   2) 시청은 1호선에도 있는 역입니다. 이 겹침이 2절과 3절에서 그대로 드러납니다
#   3) stop_order 는 여기서도 1 부터 다시 셉니다
line2 = {"name": "2", "station_count": 52, "length_km": 60.2,
         "first_station": "시청", "last_station": "까치산"}

stations2 = ["시청", "을지로입구", "을지로3가", "을지로4가",
             "동대문역사문화공원", "신당", "상왕십리", "왕십리"]

serves2 = [
    ("시청", 1, 0.0), ("을지로입구", 2, 0.7), ("을지로3가", 3, 1.5), ("을지로4가", 4, 2.1),
    ("동대문역사문화공원", 5, 3.1), ("신당", 6, 4.0), ("상왕십리", 7, 4.9), ("왕십리", 8, 5.7),
]

next2 = [
    ("시청", "을지로입구", 0.7, "01:30"), ("을지로입구", "을지로3가", 0.8, "01:00"),
    ("을지로3가", "을지로4가", 0.6, "01:00"), ("을지로4가", "동대문역사문화공원", 1.0, "01:30"),
    ("동대문역사문화공원", "신당", 0.9, "01:30"), ("신당", "상왕십리", 0.9, "01:00"),
    ("상왕십리", "왕십리", 0.8, "01:00"),
]

print(f"역 {len(set(stations1) | set(stations2))}곳(시청은 두 호선에 겹칩니다) · "
      f"운행 관계 {len(serves1) + len(serves2)}개 · 인접 관계 {len(next1) + len(next2)}개")

역 17곳(시청은 두 호선에 겹칩니다) · 운행 관계 18개 · 인접 관계 16개


---
# 1. 왜 제약부터 설계할까요?

적재에서 가장 먼저 하는 일은 데이터를 넣는 것이 아닙니다. **무엇을 같은 것으로 볼지 정하는 일**입니다. 그 판단이 곧 키이고, 키를 정하면 제약이 따라 나옵니다.

제약을 먼저 거는 이유는 그때가 가장 싸기 때문입니다. 빈 그래프에는 한 줄이면 걸리지만, 데이터가 들어간 뒤에는 **중복이 하나라도 있으면 제약 생성 자체가 실패합니다.** 어디가 중복인지 찾아 정리하는 일이 앞에 붙습니다. `NODE KEY` 는 한 가지가 더 있어서, 그 속성이 빠진 노드가 남아 있어도 걸리지 않습니다.

## 무엇을 키로 삼을까요?

원본 자료에는 칸이 여럿입니다. 그중 무엇이 **역을 가리키는 이름표**인지 정해야 합니다. 후보가 둘 있습니다.

| 후보 | 값의 예 | 골랐을 때 |
|---|---|---|
| `연번` | 1, 2, 3 … 279 | 파일 안에서 값이 겹치지 않습니다 |
| `역명` | 서울역, 시청, 종각 | 환승역은 여러 호선에 **같은 이름**으로 나옵니다 |

겹치지 않는다는 이유만 보면 `연번` 이 안전해 보입니다. 그런데 원본에서 **시청은 세 줄**입니다. 1호선의 두 번째 역으로 한 줄, 2호선의 첫 번째 역으로 한 줄, 그리고 2호선이 순환선이라 한 바퀴 돌아온 자리에 또 한 줄입니다. 세 줄의 `연번` 은 모두 다릅니다.

`연번` 을 키로 잡으면 시청은 **노드 세 개**가 됩니다. 하지만 우리가 아는 시청역은 **하나**입니다. 1호선에서 내려 2호선으로 갈아탈 수 있는 것은 그것이 같은 역이기 때문입니다. 노드가 갈리면 그 환승은 그래프에서 사라집니다.

### 규칙: 키는 파일이 아니라 현실에서 고릅니다
```text
연번을 키로  ->  시청(연번 2) 과 시청(연번 11) 이 서로 다른 노드가 된다
역명을 키로  ->  시청은 하나. 1호선과 2호선이 그 하나를 함께 가리킨다
```
키는 **파일에서 겹치지 않는 값**이 아니라 **현실에서 같은 것을 같다고 말해 주는 값**으로 고릅니다. 그래서 우리는 `역명` 을 키로 잡습니다.

> 그런데 지난 단원에서는 **"키는 이름이 아니라 `id`"** 라고 배웠습니다. 오늘은 왜 이름을 키로 쓸까요? 겹치는 상황이 세 가지로 갈리기 때문입니다.

### 이름이 겹치는 세 가지 경우

| 겹침 | 예 | 이름을 키로 쓰면 |
|---|---|---|
| 다른 종류가 같은 이름 | 지난 단원의 `Cholecalciferol`(약물이면서 약효분류) | **사고.** 관계가 한 노드에 뒤섞인다 |
| 같은 실체가 여러 줄 | 1호선과 2호선의 `시청` | **옳다.** 합쳐져야 진짜 환승역이 된다 |
| 같은 종류, 다른 실체 | 5호선과 경의중앙선의 `양평` | **사고.** 다른 역 둘이 한 노드가 된다 |

지난 단원에서 `id` 를 쓴 것은 첫 줄 때문이었습니다. 오늘 자료는 **가운데 줄**입니다. 이름이 겹치는 35곳 가운데 34곳은 진짜 환승역이고, 나머지 한 곳인 응암은 6호선 순환 구간이라 같은 호선에 두 번 적힌 것입니다. 둘 다 같은 역이니 합쳐지는 것이 옳습니다.

문제는 셋째 줄입니다. 이 자료는 서울교통공사 1~8호선만 담고 있어서 그런 역이 없지만, 코레일 노선까지 넓히면 **양평역**이 5호선과 경의중앙선에 따로 있습니다. 서로 다른 역이고 갈아탈 수 없는데, 이름을 키로 두면 한 노드로 합쳐집니다.

### 그럴 때는 이렇게 합니다

**1. 공식 식별자가 있으면 그것을 키로 삼고, 이름은 표시용 속성으로 내립니다.**

지하철에는 기관이 부여한 **역 코드**가 있습니다. 오늘 쓰는 파일에는 그 칸이 없지만, 노선을 넓힐 때는 역 코드가 든 자료를 받아 키를 옮깁니다.

```cypher
MERGE (s:Station {station_id: $station_id})   // 키는 공식 코드
SET s.name = $name                            // 이름은 사람이 읽는 표시
```

**2. 공식 식별자가 없으면 "무엇이 같으면 같은 역인가" 를 정해 키를 만듭니다.**

양평역 둘을 가르는 것은 **위치**입니다. 그러면 이름만 쓰지 말고 위치까지 함께 키로 삼습니다. 서울 양평역과 경기 양평역은 위치가 다르니 **다른 키**가 되고, 1호선 시청과 2호선 시청은 위치가 같으니 **같은 키**가 되어 하나로 합쳐집니다. 무엇을 기준으로 가를지는 데이터가 말해 주지 않고, **그 분야를 아는 사람이 정합니다.**

**3. 제약은 키를 따라갑니다.**

오늘은 이름이 키라서 이름에 제약을 겁니다. 키를 역 코드로 옮기면 **제약도 역 코드로 옮깁니다.** 이름에 걸어 둔 제약을 그대로 두면, 같은 이름의 다른 역이 들어오는 순간 적재가 통째로 막힙니다.

> **키가 안전한지는 자료의 범위가 정합니다.** 오늘 자료 안에서는 이름이 곧 역이라 이름을 키로 씁니다. 자료를 넓히는 순간 1번으로 옮기면 됩니다.

### 문법: 키 제약을 걸면 인덱스가 딸려 옵니다
```cypher
CREATE CONSTRAINT 제약이름 IF NOT EXISTS
FOR (s:Station) REQUIRE s.name IS NODE KEY
```
- **`NODE KEY`** 는 유일성과 존재를 **한꺼번에** 요구합니다. 키에는 이것이 맞습니다.
- `UNIQUE` 는 "값이 있으면 겹치지 마라" 일 뿐이라 **이름 속성이 아예 없는 노드는 막지 못합니다.**
- `IF NOT EXISTS` 를 붙이면 이미 있을 때 조용히 넘어갑니다. 노트북을 다시 돌려도 안전합니다.
- 제약에는 **이름을 붙입니다.** 나중에 목록에서 알아보고 지울 수 있습니다.
- 제약은 **인덱스를 겸합니다.** 따로 인덱스를 만들 필요가 없습니다.

> `IS NODE KEY` 는 **Enterprise 에디션**에서만 됩니다. 커뮤니티 에디션이면 이 노트북에서 이 문법을 쓰는 **코드 셀 네 곳**이 모두 같은 에러를 냅니다. `환경_구축_가이드` 대로 **Neo4j Desktop**(Enterprise 개발자 라이선스 포함)으로 실습하세요. 커뮤니티에서 그대로 가려면 네 곳을 `IS UNIQUE` 로 바꿔야 하고, 그러면 위에서 본 대비는 확인할 수 없습니다.

In [9]:
# 역과 호선에 키 제약을 건다. 아직 데이터는 하나도 넣지 않았다
run_cypher("CREATE CONSTRAINT station_name IF NOT EXISTS "
           "FOR (s:Station) REQUIRE s.name IS NODE KEY")
run_cypher("CREATE CONSTRAINT line_name IF NOT EXISTS "
           "FOR (l:Line) REQUIRE l.name IS NODE KEY")
# 걸린 제약을 이름과 종류로 확인한다
for c in run_cypher("SHOW CONSTRAINTS YIELD name, type, labelsOrTypes "
                    "RETURN name, type, labelsOrTypes"):
    print(c)

{'name': 'line_name', 'type': 'NODE_KEY', 'labelsOrTypes': ['Line']}
{'name': 'station_name', 'type': 'NODE_KEY', 'labelsOrTypes': ['Station']}


> 레이블마다 따로 걸었다는 점을 보세요. `:Station` 에 건 제약은 `:Line` 을 지켜 주지 않습니다. **제약도 인덱스처럼 레이블별**입니다.

### 따라하기에서 쓸 자료: 의료 지식그래프

이 단원의 `🖐️ 함께 따라하기` 는 **다른 데이터**로 같은 절차를 밟습니다. 지난 단원에서 쓰던 의료 지식그래프의 작은 조각입니다.

<img src="images/의료그래프_조각.png" width="860">

*약물이 질병을 치료하는 관계 하나뿐인 단순한 그래프입니다. 그런데 **키가 이름이 아니라 `id`** 입니다.*

왜 여기서는 `id` 일까요. 위 「이름이 겹치는 세 가지 경우」 표의 **첫째 줄**에 해당하는 자료이기 때문입니다. 약물 `Cholecalciferol` 과 약효분류 `Cholecalciferol` 처럼 **다른 종류가 같은 이름**을 쓰는 일이 있어서, 이름으로 합치면 서로 다른 것이 한 노드가 됩니다.

**같은 절차, 다른 키.** 지하철에서 `name` 에 걸었던 제약을 여기서는 `id` 에 겁니다.

In [10]:
# [제공 코드] 따라하기용 조각(의료 지식그래프): 이 셀은 실행만 하세요(값 정의).
# 지난 단원에서 쓰던 그 의료 그래프의 작은 조각입니다. 지하철과 달리 **id 가 키**입니다.
# 약물과 약효분류가 같은 이름을 쓸 수 있어서, 이름으로는 서로를 가릴 수 없기 때문입니다

# [노드] :Compound 약물 · :Disease 질병. 둘 다 같은 두 값을 갖습니다
#          id   : 원본 자료의 식별자(문자열). 이 값이 키입니다
#          name : 사람이 읽는 이름(문자열). 키가 아닙니다
compounds = [   # 약물
    {"id": "Compound::DB00661", "name": "Verapamil"},
    {"id": "Compound::DB00571", "name": "Propranolol"},
    {"id": "Compound::DB00177", "name": "Valsartan"},
    {"id": "Compound::DB00744", "name": "Zileuton"},
    {"id": "Compound::DB00549", "name": "Zafirlukast"},
    {"id": "Compound::DB00313", "name": "Valproic Acid"},
]

diseases = [   # 질병
    {"id": "Disease::DOID:10763", "name": "hypertension"},
    {"id": "Disease::DOID:2841", "name": "asthma"},
    {"id": "Disease::DOID:6364", "name": "migraine"},
]

# [관계] (:Compound)-[:TREATS]->(:Disease) - 이 약물이 그 질병을 치료합니다
#        자리는 (약물 id, 질병 id). 붙는 속성은 없습니다
treats = [
    ("Compound::DB00661", "Disease::DOID:10763"),   # Verapamil -> hypertension
    ("Compound::DB00661", "Disease::DOID:6364"),    # Verapamil -> migraine
    ("Compound::DB00571", "Disease::DOID:10763"),   # Propranolol -> hypertension
    ("Compound::DB00571", "Disease::DOID:6364"),    # Propranolol -> migraine
    ("Compound::DB00177", "Disease::DOID:10763"),   # Valsartan -> hypertension
    ("Compound::DB00744", "Disease::DOID:2841"),    # Zileuton -> asthma
    ("Compound::DB00549", "Disease::DOID:2841"),    # Zafirlukast -> asthma
    ("Compound::DB00313", "Disease::DOID:6364"),    # Valproic Acid -> migraine
]

print(f"약물 {len(compounds)}개 · 질병 {len(diseases)}개 · 치료 관계 {len(treats)}개")

약물 6개 · 질병 3개 · 치료 관계 8개


### 🖐️ 함께 따라하기: 의료 지식그래프에 제약 걸기

아직 노드를 하나도 넣지 않았습니다. **지금이 제약을 걸 자리**입니다.

1. `Compound` 의 **`id`** 에 키 제약을 겁니다. 제약 이름은 `compound_id` 로 합니다.
2. `Disease` 의 **`id`** 에도 같은 모양으로 겁니다. 이름은 `disease_id` 입니다.
3. `SHOW CONSTRAINTS` 로 지금까지 걸린 제약을 **이름순으로**, 이름과 종류까지 모두 출력합니다.

In [15]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) CREATE CONSTRAINT compound_id IF NOT EXISTS
#    FOR (c:Compound) REQUIRE c.id IS NODE KEY
# 2) disease_id 도 같은 모양으로 (:Disease) 의 id 에 건다
# 3) SHOW CONSTRAINTS YIELD name, type, labelsOrTypes 를 이름순으로 출력한다

run_cypher("CREATE CONSTRAINT compound_id IF NOT EXISTS "
           "FOR (c:Compound) REQUIRE c.id IS NODE KEY")


run_cypher("CREATE CONSTRAINT disease_id IF NOT EXISTS "
           "FOR (d:Disease) REQUIRE d.id IS NODE KEY")

# 걸린 제약을 이름과 종류로 확인한다
for c in run_cypher("SHOW CONSTRAINTS YIELD name, type, labelsOrTypes "
                    "RETURN name, type, labelsOrTypes"):
    print(c)



{'name': 'compound_id', 'type': 'NODE_KEY', 'labelsOrTypes': ['Compound']}
{'name': 'disease_id', 'type': 'NODE_KEY', 'labelsOrTypes': ['Disease']}
{'name': 'line_name', 'type': 'NODE_KEY', 'labelsOrTypes': ['Line']}
{'name': 'station_name', 'type': 'NODE_KEY', 'labelsOrTypes': ['Station']}


> 네 개가 걸렸습니다. **레이블마다, 그리고 그 레이블의 키마다** 하나씩입니다. 지하철은 `name`, 의료는 `id` 로 서로 다른 속성에 걸렸다는 점을 보세요.

In [12]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) CREATE CONSTRAINT compound_id IF NOT EXISTS
#    FOR (c:Compound) REQUIRE c.id IS NODE KEY

# 2) disease_id 도 같은 모양으로 (:Disease) 의 id 에 건다
# 3) SHOW CONSTRAINTS YIELD name, type, labelsOrTypes 를 이름순으로 출력한다


# 약품 id 제약조건
run_cypher("CREATE CONSTRAINT compound_id IF NOT EXISTS "
           "FOR (c:Compound) REQUIRE c.id IS NODE KEY")

# 질병 id 제약조건
run_cypher("CREATE CONSTRAINT disease_id IF NOT EXISTS "
           "FOR (d:Disease) REQUIRE d.id IS NODE KEY")

# 걸린 제약을 이름과 종류로 확인한다
for c in run_cypher("SHOW CONSTRAINTS YIELD name, type, labelsOrTypes "
                    "RETURN name, type, labelsOrTypes"):
    print(c)

{'name': 'compound_id', 'type': 'NODE_KEY', 'labelsOrTypes': ['Compound']}
{'name': 'disease_id', 'type': 'NODE_KEY', 'labelsOrTypes': ['Disease']}
{'name': 'line_name', 'type': 'NODE_KEY', 'labelsOrTypes': ['Line']}
{'name': 'station_name', 'type': 'NODE_KEY', 'labelsOrTypes': ['Station']}


### ✅ 바로 확인 퀴즈

**1)** 파일 안에서 값이 한 번도 겹치지 않는 칸이 있습니다. 그 칸을 키로 삼으면 항상 안전할까요?

<details><summary>정답 보기</summary>

아닙니다. 파일에서 겹치지 않는 것과 **현실에서 다른 것**은 별개입니다. `연번` 은 값이 겹치지 않지만, 같은 역이 호선마다 다른 `연번` 을 갖기 때문에 그것을 키로 삼으면 **한 역이 여러 노드로 갈립니다.** 키는 현실에서 같은 것을 같다고 말해 주는 값으로 고릅니다.

</details>

**2)** `UNIQUE` 와 `NODE KEY` 는 무엇이 다른가요?

<details><summary>정답 보기</summary>

`UNIQUE` 는 **값이 있을 때 겹치지 않을 것**만 요구합니다. 그래서 그 속성이 아예 없는 노드는 몇 개든 들어옵니다. `NODE KEY` 는 여기에 **반드시 있을 것**을 더합니다. 키에는 `NODE KEY` 가 맞고, 에디션 때문에 쓸 수 없을 때만 `UNIQUE` 로 갑니다.

</details>

**3)** `:Station` 에 제약을 걸었습니다. `:Line` 도 자동으로 보호되나요?

<details><summary>정답 보기</summary>

아닙니다. 제약은 **레이블별**로 걸립니다. 인덱스가 레이블마다 따로 걸리는 것과 같습니다. `:Line` 을 보호하려면 `:Line` 에 따로 걸어야 합니다.

</details>

---
# 2. 노드를 넣습니다

제약을 걸었으니 이제 노드를 넣습니다. 관계는 아직 손대지 않습니다. **양 끝이 다 있어야 화살표를 그을 수 있기** 때문입니다.

## 2-1. `MERGE` 에는 키만, 나머지는 `SET` 으로

### 왜 필요할까요?
`MERGE` 는 중괄호 안의 값으로 **찾아보고, 없으면 만듭니다.** 그러니 중괄호에는 **변하지 않는 식별 키**만 들어가야 합니다.

만약 `MERGE (s:Station {name: '시청', riders: 52933})` 처럼 바뀔 수 있는 값까지 넣으면, 이용객 수가 달라진 다음 실행에서 **찾지 못하고 새로 만들려 합니다.**

여기서 두 갈래로 갈립니다. 제약이 없으면 같은 역이 **노드 둘로 갈리고**, 1절에서 건 키 제약이 있으면 그 자리에서 `ConstraintValidationFailed` 가 나며 **적재가 통째로 멈춥니다.** 어느 쪽이든 사고입니다.

### 문법: 찾을 값과 채울 값을 나눕니다
```cypher
MERGE (l:Line {name: $name})           // 찾을 값: 키만
SET l.station_count = $station_count,  // 채울 값: 나머지 전부
    l.length_km = $length_km
```
호선 노드에는 자료형이 섞여 들어갑니다. 이름은 문자열, 역수는 정수, 총연장은 실수입니다. Cypher 는 넘어온 값의 자료형을 그대로 저장합니다.

In [13]:
# 호선 노드 하나. MERGE 에는 키(name)만 두고 나머지 네 개는 SET 으로 채운다
run_cypher("""
MERGE (l:Line {name: $name})
SET l.station_count = $station_count,
    l.length_km = $length_km,
    l.first_station = $first_station,
    l.last_station = $last_station
""", **line1)
print(run_cypher("MATCH (l:Line {name: '1'}) RETURN l.name AS 호선, "
                 "l.station_count AS 역수, l.length_km AS 총연장")[0])

{'호선': '1', '역수': 10, '총연장': 7.8}


> 값을 `SET` 으로 채운 이유가 있습니다. **원본이 바뀌면 그 값이 반영돼야 하기 때문**입니다. 노선이 연장되면 총연장도 따라 바뀌어야 합니다.

### 그런데 덮어쓰면 안 되는 것도 있습니다

값은 매번 덮어써야 최신이 됩니다. 그런데 **처음 들어온 때**는 다릅니다. 재적재할 때마다 덮어쓰면 그 값은 "처음" 이 아니라 "마지막" 이 되어 버립니다.

`MERGE` 는 **만든 경우와 찾은 경우를 갈라서** 다르게 처리할 수 있습니다.

```cypher
MERGE (s:Station {name: $name})
ON CREATE SET s.created_at = date($today)   // 없어서 새로 만들었을 때만
ON MATCH SET  s.updated_at = date($today)   // 이미 있어서 찾았을 때만
SET s.line_count = $count                   // 어느 경우든
```
- `ON CREATE SET` 은 **처음 만들 때만**, `ON MATCH SET` 은 **이미 있을 때만** 실행됩니다.
- 둘 다 없는 일반 `SET` 은 **어느 경우든** 실행됩니다.

<img src="images/on_create_set_갈림길.png" width="760">

같은 `MERGE` 문장이 상황에 따라 다른 길로 갑니다. 없으면 만들면서 초기값을, 있으면 찾아서 갱신값을 넣습니다. 일반 `SET` 은 어느 쪽으로 갔든 실행됩니다.

> 둘을 가르는 기준은 **그 값이 어디서 왔느냐**입니다. `created_at` 도 속성이지만 원본 파일에 있던 값이 아닙니다.

> - **원본 자료에서 온 값**(역 이름·총연장)은 일반 `SET` 으로 매번 덮어씁니다. 원본이 바뀌면 따라 바뀌어야 하니까요. 여기에 `ON CREATE SET` 을 쓰면 **처음 넣은 값에 박제되어** 원본이 갱신돼도 반영되지 않습니다.
> - **적재가 남기는 기록**(`created_at`·`updated_at`)은 그 순간에만 뜻이 있어서 나중에 덮으면 거짓이 됩니다. 그래서 `ON CREATE`·`ON MATCH` 로 갈라 씁니다. 이 두 이름은 거의 모든 데이터베이스 테이블에 있는 관례라, 다른 코드를 읽을 때도 그대로 만납니다.
> - `updated_at` 은 **값이 실제로 바뀌었는지와 무관하게** 이 노드를 마지막으로 적재한 날입니다. `MERGE` 가 찾기만 해도 갱신됩니다.

> 실무 적재 코드가 거의 늘 이 모양입니다.

In [17]:
# 역 노드 열 개를 넣는다. 적재한 날짜는 파라미터로 넘긴다(배치가 언제 돌았는지 남긴다)
# 지금은 빈 그래프라 열 개 모두 "새로 만드는" 쪽으로 간다
for name in stations1:
    run_cypher("""
MERGE (s:Station {name: $name})
ON CREATE SET s.created_at = date($today)
ON MATCH SET  s.updated_at = date($today)
""", name=name, today="2026-03-02")
print(run_cypher("MATCH (s:Station {name: '서울역'}) "
                 "RETURN s.name AS 역, s.created_at AS 생성일, "
                 "s.updated_at AS 수정일")[0])

{'역': '서울역', '생성일': neo4j.time.Date(2026, 3, 2), '수정일': neo4j.time.Date(2026, 3, 2)}


`생성일` 만 채워지고 `수정일` 은 비어 있습니다. 열 개 전부 새로 만들어졌기 때문입니다.

이제 **다음 날 같은 배치가 한 번 더 돈다고** 해 보겠습니다. 날짜만 바꿔 같은 코드를 그대로 실행합니다.

In [ ]:
# 같은 코드를 날짜만 바꿔 다시 실행한다. 이번에는 열 개 모두 "이미 있는" 쪽으로 간다
for name in stations1:
    run_cypher("""
MERGE (s:Station {name: $name})
ON CREATE SET s.created_at = date($today)
ON MATCH SET  s.updated_at = date($today)
""", name=name, today="2026-03-03")
print(run_cypher("MATCH (s:Station) "
                 "RETURN count(s) AS 역노드수, "
                 "min(s.created_at) AS 생성일, max(s.updated_at) AS 수정일")[0])

> 두 가지를 함께 보세요. 노드 수가 **늘지 않았고**(`MERGE` 라서), `생성일` 은 **3월 2일 그대로**입니다. 바뀐 것은 `수정일` 뿐입니다. 만약 처음 들어온 때를 일반 `SET` 으로 적었다면 3월 3일로 덮여, **언제 처음 들어왔는지 영영 알 수 없게** 됩니다.

---
## 2-2. 한 줄씩 보낼까요, 한 번에 묶어 보낼까요?

### 왜 필요할까요?
위에서 역 열 개를 넣으려고 **열 번** 왕복했습니다. 열 개면 괜찮지만 수만 줄이면 왕복 자체가 비용이 됩니다. 값을 **리스트로 통째로 넘기고** 데이터베이스 쪽에서 한 줄씩 풀게 하는 방법이 있습니다.

### 문법: `UNWIND` 는 리스트를 줄로 폅니다
```cypher
UNWIND $rows AS row        // 리스트를 한 줄씩 편다
MERGE (s:Station {name: row.name})
```
`$rows` 로 리스트를 한 번에 넘기면, `UNWIND` 가 그 안을 한 줄씩 꺼내 뒤 문장을 반복 실행합니다. 왕복은 **한 번**입니다. 갈림길 문법도 그대로 붙일 수 있습니다.

<img src="images/한줄씩_대_UNWIND.png" width="760">

같은 일을 하지만 왕복 횟수가 다릅니다. **순서 규칙은 어느 쪽이든 그대로**입니다. 묶어 보내도 노드가 먼저, 관계가 나중입니다.

In [21]:
# 이번에는 2호선을 넣는다. 호선 노드 하나와 역 여덟 개다
run_cypher("""
MERGE (l:Line {name: $name})
SET l.station_count = $station_count,
    l.length_km = $length_km,
    l.first_station = $first_station,
    l.last_station = $last_station
""", **line2)

# 역 여덟 개는 UNWIND 로 한 번에 보낸다
rows2 = [{"name": name} for name in stations2]
run_cypher("""
UNWIND $rows AS row
MERGE (s:Station {name: row.name})
ON CREATE SET s.created_at = date($today)
ON MATCH SET  s.updated_at = date($today)
""", rows=rows2, today="2026-03-04")
print(run_cypher("MATCH (s:Station) RETURN count(s) AS 역노드수")[0])

{'역노드수': 17}


In [19]:
# 시청은 1호선에도 있던 역이다. 어느 갈래로 갔는지 확인한다
print(run_cypher("MATCH (s:Station {name: '시청'}) "
                 "RETURN s.created_at AS 생성일, s.updated_at AS 수정일")[0])

{'생성일': neo4j.time.Date(2026, 3, 2), '수정일': neo4j.time.Date(2026, 3, 4)}


> 2호선 역이 여덟 개인데 전체 역 수는 열여덟이 아닙니다. **시청이 1호선에 이미 있어서** `MERGE` 가 새로 만들지 않았기 때문입니다. 시청의 `생성일` 이 3월 2일인 것이 그 증거입니다. 이번에 처음 만들어진 나머지 일곱 개는 `생성일` 이 3월 4일입니다.

### 🖐️ 함께 따라하기: 의료 지식그래프 노드 넣기

1절에서 제약은 이미 걸어 두었습니다. 이제 **노드를 넣을 차례**입니다.

1. `compounds` 와 `diseases` 를 각각 `UNWIND` 로 넣습니다. `MERGE` 는 **`id`** 로 하고, 그 바로 뒤에 `ON CREATE SET`·`ON MATCH SET` 을 붙인 다음 `name` 을 일반 `SET` 으로 채웁니다(날짜는 `'2026-03-05'`).

> 이 순서를 바꾸면 문법 오류입니다. `ON CREATE`·`ON MATCH` 는 `MERGE` 에 딸린 절이라 일반 `SET` 뒤에는 올 수 없습니다.
2. 약물 수를 `n_compound`, 질병 수를 `n_disease` 에 담아 두 줄로 출력합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) UNWIND $rows AS row
#    MERGE (c:Compound {id: row.id})
#    ON CREATE SET c.created_at = date($today)
#    ON MATCH SET  c.updated_at = date($today)
#    SET c.name = row.name
#    절 순서를 이대로 지킨다. ON CREATE / ON MATCH 는 MERGE 바로 뒤에만 올 수 있다
#    diseases 도 같은 방법으로 (:Disease) 에 넣는다
# 2) count 로 n_compound, n_disease 를 구해 두 줄로 출력한다

In [24]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) UNWIND $rows AS row
#    MERGE (c:Compound {id: row.id})
#    ON CREATE SET c.created_at = date($today)
#    ON MATCH SET  c.updated_at = date($today)
#    SET c.name = row.name
#    절 순서를 이대로 지킨다. ON CREATE / ON MATCH 는 MERGE 바로 뒤에만 올 수 있다
#    diseases 도 같은 방법으로 (:Disease) 에 넣는다

run_cypher("""
UNWIND $rows AS row
MERGE (c:Compound {id: row.id})
ON CREATE SET c.created_at = date($today)
ON MATCH SET  c.updated_at = date($today)
SET c.name = row.name
""", rows=compounds, today='2026-09-01')

run_cypher("""
UNWIND $rows AS row
MERGE (d:Disease {id: row.id})
ON CREATE SET d.created_at = date($today)
ON MATCH SET  d.updated_at = date($today)
SET d.name = row.name
""", rows=diseases, today='2026-09-01')


# 2) count 로 n_compound, n_disease 를 구해 두 줄로 출력한다
print(run_cypher("""MATCH (c:Compound) RETURN count(c) as n""")[0]['n']) # 6
print(run_cypher("""MATCH (d:Disease) RETURN count(d) as n""")[0]['n']) # 3


6
3


> 같은 절차인데 키만 다릅니다. 지하철은 이름이 곧 역이라 `name` 을 키로 썼고, 의료 자료는 약물과 약효분류가 같은 이름을 쓸 수 있어 `id` 를 키로 썼습니다. **키는 자료를 보고 정한다**는 1절의 이야기가 여기서 그대로 되풀이됩니다.

### ✅ 바로 확인 퀴즈

**1)** `MERGE (s:Station {name: '시청', riders: 52933})` 처럼 중괄호에 이용객 수까지 넣으면 무엇이 나빠지나요?

<details><summary>정답 보기</summary>

이용객 수는 바뀌는 값입니다. 다음에 다른 수로 실행하면 `MERGE` 가 그 조합을 **못 찾고 새로 만들려 합니다.** 제약이 없으면 같은 역이 둘로 갈리고, 키 제약이 있으면 거기서 적재가 멈춥니다. 중괄호에는 변하지 않는 식별 키만 넣고, 나머지는 `SET` 으로 채웁니다.

</details>

**2)** 총연장을 `ON CREATE SET` 에 두면 무엇이 문제인가요? `created_at` 도 속성인데 왜 그쪽은 `ON CREATE SET` 이 맞나요?

<details><summary>정답 보기</summary>

총연장은 **원본 자료에서 온 값**입니다. 노선이 연장되어 원본이 바뀌면 그래프도 따라 바뀌어야 하는데, `ON CREATE SET` 에 두면 다시 적재할 때 `ON MATCH` 쪽으로 가므로 **처음 넣은 값에 박제됩니다.**

`created_at` 은 원본에 있던 값이 아니라 **적재가 남기는 기록**입니다. "이 노드를 처음 만든 순간" 이라는 사실이라, 나중에 덮으면 그 뜻 자체가 거짓이 됩니다. 둘 다 속성이지만 **값이 어디서 왔느냐**가 다릅니다.

</details>

**3)** `UNWIND` 로 묶어 보내면 적재 순서 규칙이 달라지나요?

<details><summary>정답 보기</summary>

달라지지 않습니다. 왕복 횟수만 줄어들 뿐, **노드를 다 넣은 다음 관계**라는 순서는 그대로 지켜야 합니다. 한 번에 보내든 나눠 보내든 관계를 그으려면 양 끝이 이미 있어야 합니다.

</details>

---
# 3. 관계를 잇습니다

노드가 다 들어왔으니 이제 화살표를 긋습니다. 관계를 만들 때 지킬 것은 하나입니다. **양 끝을 같은 질의 안에서 찾아야 합니다.**

## 3-1. 양 끝을 `MATCH` 로 찾아 잇습니다

### 왜 필요할까요?
관계는 두 노드를 잇는 화살표입니다. 그런데 앞 셀에서 노드를 만들 때 쓴 변수 이름은 **그 질의가 끝나는 순간 사라집니다.** 새 질의에서 같은 글자를 써도 그것은 처음 보는 이름입니다.

그래서 관계를 그을 때는 **같은 질의 안에서 `MATCH` 로 양 끝을 다시 찾아** 잡습니다.

### 문법: 펴고, 찾고, 잇습니다
```cypher
UNWIND $rows AS row                                                // 리스트를 한 줄씩 펴고
MATCH (l:Line {name: row.line}), (s:Station {name: row.station})   // 양 끝을 찾고
MERGE (l)-[r:SERVES]->(s)                                          // 그 둘을 잇는다
```
노드에 쓴 `UNWIND` 를 관계에도 그대로 씁니다. 양 끝 이름을 리스트에 담아 한 번에 보내고, 펴진 줄마다 `MATCH` 가 양 끝을 찾습니다.

`MATCH` 에 **레이블을 꼭 적으세요.** 1절에서 건 제약이 인덱스를 겸하는데, 레이블을 적어야 그 인덱스를 씁니다. 레이블 없이 `MATCH (l {name: ...})` 라고 쓰면 전체를 훑습니다.

In [26]:
serves_rows = [{"line": ln, "station": st, "stop_order": order, "cumulative_km": km}
               for ln, serves in (("1", serves1), ("2", serves2))
               for st, order, km in serves]

print(serves_rows)

[{'line': '1', 'station': '서울역', 'stop_order': 1, 'cumulative_km': 0.0}, {'line': '1', 'station': '시청', 'stop_order': 2, 'cumulative_km': 1.1}, {'line': '1', 'station': '종각', 'stop_order': 3, 'cumulative_km': 2.1}, {'line': '1', 'station': '종로3가', 'stop_order': 4, 'cumulative_km': 2.9}, {'line': '1', 'station': '종로5가', 'stop_order': 5, 'cumulative_km': 3.8}, {'line': '1', 'station': '동대문', 'stop_order': 6, 'cumulative_km': 4.6}, {'line': '1', 'station': '동묘앞', 'stop_order': 7, 'cumulative_km': 5.2}, {'line': '1', 'station': '신설동', 'stop_order': 8, 'cumulative_km': 5.9}, {'line': '1', 'station': '제기동', 'stop_order': 9, 'cumulative_km': 6.8}, {'line': '1', 'station': '청량리', 'stop_order': 10, 'cumulative_km': 7.8}, {'line': '2', 'station': '시청', 'stop_order': 1, 'cumulative_km': 0.0}, {'line': '2', 'station': '을지로입구', 'stop_order': 2, 'cumulative_km': 0.7}, {'line': '2', 'station': '을지로3가', 'stop_order': 3, 'cumulative_km': 1.5}, {'line': '2', 'station': '을지로4가', 'stop_order': 4, 'cumulat

In [27]:
# 호선 -> 역 관계. UNWIND 로 묶어 보내고, 그 안에서 양 끝을 MATCH 로 찾아 MERGE 로 잇는다
# stop_order(그 호선에서 몇 번째로 서는지)와 cumulative_km(기점부터 누계)를 관계에 매단다
serves_rows = [{"line": ln, "station": st, "stop_order": order, "cumulative_km": km}
               for ln, serves in (("1", serves1), ("2", serves2))
               for st, order, km in serves]
run_cypher("""
UNWIND $rows AS row
MATCH (l:Line {name: row.line}), (s:Station {name: row.station})
MERGE (l)-[r:SERVES]->(s)
SET r.stop_order = row.stop_order, r.cumulative_km = row.cumulative_km
""", rows=serves_rows)
print(run_cypher("MATCH (:Line)-[r:SERVES]->(:Station) "
                 "RETURN count(r) AS 운행관계수")[0])

{'운행관계수': 18}


In [28]:
# 관계에 매단 값이 제대로 붙었는지 한 줄만 꺼내 본다
print(run_cypher("""
MATCH (l:Line {name: '2'})-[r:SERVES]->(s:Station {name: '을지로3가'})
RETURN s.name AS 역, r.stop_order AS 정차순서, r.cumulative_km AS 기점부터
""")[0])

{'역': '을지로3가', '정차순서': 3, '기점부터': 1.5}


---
## 3-2. 역 사이의 값은 관계에 매답니다

### 왜 필요할까요?
역간거리 1.1km 는 **서울역의 성질도 시청의 성질도 아닙니다.** 두 역 사이의 값입니다. 이런 값을 어느 한쪽 노드에 넣으면 어느 쪽에 넣을지부터 애매해지고, 나중에 그 역에 이웃이 하나 더 생기면 값이 충돌합니다.

관계에 매달면 그 문제가 없습니다. 화살표마다 자기 거리와 자기 시간을 갖습니다.

### 문법: 관계에도 변수를 주고 `SET` 합니다
```cypher
UNWIND $rows AS row
MATCH (a:Station {name: row.src}), (b:Station {name: row.dst})
MERGE (a)-[r:NEXT {line: row.line}]->(b)   // 관계에 변수 r 을 준다
SET r.distance_km = row.distance_km, r.travel_time = row.travel_time
```
`NEXT` 의 중괄호에 `line` 을 넣은 것을 보세요. **어느 호선의 인접인지**가 이 관계를 구분하는 키입니다. 같은 두 역이 여러 호선으로 이어질 수 있기 때문입니다.

> 오늘 넣는 조각에는 그런 구간이 없어서 `line` 을 빼도 관계 수가 같습니다. 하지만 원본 파일 전체에는 딱 하나 있습니다. **을지로4가에서 동대문역사문화공원**으로 가는 구간이 2호선과 5호선 양쪽에 있습니다. `line` 을 빼면 이 둘이 한 관계로 합쳐져 한쪽의 거리와 시간이 사라집니다. 조각에서 안 보인다고 빼 두면 파일 전체를 넣는 순간 값을 잃습니다.

In [29]:
# 역 -> 역 인접 관계. 거리(실수)와 소요시간(문자열)을 관계 속성으로 넣는다
next_rows = [{"line": ln, "src": a, "dst": b, "distance_km": km, "travel_time": t}
             for ln, nexts in (("1", next1), ("2", next2))
             for a, b, km, t in nexts]
run_cypher("""
UNWIND $rows AS row
MATCH (a:Station {name: row.src}), (b:Station {name: row.dst})
MERGE (a)-[r:NEXT {line: row.line}]->(b)
SET r.distance_km = row.distance_km, r.travel_time = row.travel_time
""", rows=next_rows)
print(run_cypher("MATCH (:Station)-[r:NEXT]->(:Station) "
                 "RETURN count(r) AS 인접관계수")[0])

{'인접관계수': 16}


In [30]:
# 넣은 값이 관계에 제대로 붙었는지 한 구간만 꺼내 본다
print(run_cypher("""
MATCH (a:Station {name: '서울역'})-[r:NEXT]->(b:Station)
RETURN a.name AS 출발, b.name AS 도착, r.distance_km AS 거리, r.travel_time AS 소요시간
""")[0])

{'출발': '서울역', '도착': '시청', '거리': 1.1, '소요시간': '02:00'}


### 🖐️ 함께 따라하기: 약물이 치료하는 질병 잇기

앞에서 넣은 약물과 질병을 이어 보세요. 노드는 이미 다 있으니 **찾아서 잇기만** 하면 됩니다.

1. `treats` 를 `[{"src": ..., "dst": ...}, ...]` 로 만들어 `$rows` 로 넘기고, `UNWIND` 안에서 `MATCH` 로 양 끝을 찾아 `(:Compound)-[:TREATS]->(:Disease)` 를 `MERGE` 합니다. 양 끝은 **`id`** 로 찾습니다.
2. 만들어진 `TREATS` 관계 수를 `n_treats` 에 담아 출력합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) treats 를 [{"src": ..., "dst": ...}, ...] 로 만들어 rows 로 넘긴다
# 2) UNWIND $rows AS row 뒤에 MATCH (c:Compound {id: row.src}), (d:Disease {id: row.dst})
#    로 찾고 MERGE (c)-[:TREATS]->(d) 로 잇는다
# 3) MATCH (:Compound)-[r:TREATS]->(:Disease) RETURN count(r) 로 n_treats 를 구해 출력한다

In [34]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) treats 를 [{"src": ..., "dst": ...}, ...] 로 만들어 rows 로 넘긴다
# 2) UNWIND $rows AS row 뒤에 MATCH (c:Compound {id: row.src}), (d:Disease {id: row.dst})
#    로 찾고 MERGE (c)-[:TREATS]->(d) 로 잇는다
# 3) MATCH (:Compound)-[r:TREATS]->(:Disease) RETURN count(r) 로 n_treats 를 구해 출력한다
rows = [{"src":src, "dst":dst} for src, dst in treats]
print(rows)

run_cypher("""
UNWIND $rows AS row
MATCH (c:Compound {id: row.src}), (d:Disease {id: row.dst})
MERGE (c)-[:TREATS]->(d)
""", rows=rows)

print(run_cypher("MATCH (:Compound)-[r:TREATS]->(:Disease) RETURN count(r)"))

[{'src': 'Compound::DB00661', 'dst': 'Disease::DOID:10763'}, {'src': 'Compound::DB00661', 'dst': 'Disease::DOID:6364'}, {'src': 'Compound::DB00571', 'dst': 'Disease::DOID:10763'}, {'src': 'Compound::DB00571', 'dst': 'Disease::DOID:6364'}, {'src': 'Compound::DB00177', 'dst': 'Disease::DOID:10763'}, {'src': 'Compound::DB00744', 'dst': 'Disease::DOID:2841'}, {'src': 'Compound::DB00549', 'dst': 'Disease::DOID:2841'}, {'src': 'Compound::DB00313', 'dst': 'Disease::DOID:6364'}]
[{'count(r)': 8}]


---
## 3-3. 다 이었으면 세어서 넣습니다

### 왜 필요할까요?
관계가 다 들어왔으니 이제 **관계를 세어야 알 수 있는 것**을 노드에 적어 둘 수 있습니다. 대표적인 것이 **환승역**입니다. 어떤 역이 환승역인지는 역 하나만 봐서는 알 수 없고, 그 역에 붙은 호선이 몇 개인지로 알 수 있습니다.

### 문법: 세어서 그 값을 그대로 넣습니다
```cypher
MATCH (l:Line)-[:SERVES]->(s:Station)
WITH s, count(l) AS line_count      // 역마다 붙은 호선 수를 센다
SET s.line_count = line_count,      // 센 값을 그대로 넣는다
    s.transfer = line_count >= 2    // 둘 이상이면 환승역
```
`transfer` 는 불리언입니다. 문자열·정수·실수·날짜에 이어 자료형이 하나 더 늘었습니다.

이 방식은 **몇 번을 다시 돌려도 결과가 같습니다.** 매번 관계를 새로 세어 덮어쓰기 때문입니다. `s.line_count = s.line_count + 1` 처럼 **누적하면 다시 돌릴 때마다 늘어나** 멱등이 깨집니다.

In [31]:
# 역마다 지나는 호선 수를 세어 역 속성으로 넣는다. 다시 돌려도 같은 값이 나온다
run_cypher("""
MATCH (l:Line)-[:SERVES]->(s:Station)
WITH s, count(l) AS line_count
SET s.line_count = line_count, s.transfer = line_count >= 2
""")
for r in run_cypher("MATCH (s:Station) WHERE s.transfer "
                    "RETURN s.name AS 역, s.line_count AS 지나는호선수"):
    print(r)

{'역': '시청', '지나는호선수': 2}


> 오늘 넣은 두 조각에서 겹치는 역은 시청 하나입니다. 원본 파일 전체를 넣으면 환승역이 훨씬 많아지는데, 그 규모는 **교안_03** 에서 확인합니다.

### ✅ 바로 확인 퀴즈

**1)** 관계를 그을 때 `MATCH` 에서 레이블을 빼고 `MATCH (a {name: $src})` 라고 쓰면 무엇이 달라지나요?

<details><summary>정답 보기</summary>

결과는 같을 수 있지만 **느려집니다.** 제약이 겸하는 인덱스는 `:Station` 이라는 레이블에 걸려 있어서, 레이블을 적어야 그 인덱스를 씁니다. 레이블이 없으면 그래프의 모든 노드를 훑습니다.

</details>

**2)** 역간거리를 `Station` 노드의 속성으로 두면 무엇이 곤란해지나요?

<details><summary>정답 보기</summary>

거리는 **두 역 사이의 값**이라 어느 한 역에 넣으면 어느 쪽에 넣을지부터 애매합니다. 게다가 한 역에 이웃이 여럿 생기면(환승역이나 분기역) 값이 하나뿐인 속성 자리에 여러 거리를 넣을 수 없습니다. 관계에 매달면 화살표마다 자기 값을 갖습니다.

</details>

**3)** 호선 수를 `ON CREATE SET s.line_count = 1` · `ON MATCH SET s.line_count = s.line_count + 1` 처럼 **더해서** 세면 무엇이 문제가 되나요?

<details><summary>정답 보기</summary>

적재를 다시 돌릴 때마다 값이 계속 늘어납니다. 시청은 호선 둘이 지나므로 한 번 돌리면 2, 두 번 돌리면 4 가 됩니다. **누적하는 값은 멱등하지 않습니다.** 그래서 3-3 처럼 매번 관계를 새로 세어 **덮어쓰는** 방식이 안전합니다.

`ON CREATE SET` 을 빠뜨리면 더 조용한 사고가 납니다. 값이 없는 상태에서 `s.line_count + 1` 은 `null` 이고, `null` 을 `SET` 하면 속성이 아예 생기지 않습니다. 에러 없이 빈 채로 남습니다.

</details>

---
## 전체 적재 코드 한눈에 보기

절마다 나눠 실행한 것을 **하나로 모으면** 이런 모양입니다. 실무의 적재 스크립트가 바로 이렇게 생겼습니다. 제약을 먼저 걸고, 노드를 전부 넣고, 관계를 잇고, 마지막에 세어서 채웁니다.

**자료 출처가 다르면 스크립트도 나눕니다.** 지하철과 의료는 키도 다르고 갱신 주기도 다르므로 각각 하나씩 만듭니다.

이미 다 들어 있는 그래프에 그대로 다시 돌립니다. **아무것도 늘지 않아야** 맞습니다.

> 이 절은 **앞 절을 모두 실행한 상태**를 전제합니다. 따라하기를 비워 두었다면 의료 노드가 여기서 처음 들어가므로 수가 늘어나고, 초기화 직후 이 절부터 실행하면 아직 관계가 없어 아래 셀이 에러를 냅니다. 앞 절을 먼저 채우고 오세요.

In [ ]:
# 다시 돌리기 전에 지금 그래프 크기를 재 둔다
print(run_cypher("""
MATCH (n) WITH count(n) AS 노드
MATCH ()-[r]->() RETURN 노드, count(r) AS 관계
""")[0])

### 지하철 적재

In [ ]:
# 적재한 날을 기록으로 남긴다. 실무 배치는 돌린 그날이 들어간다
from datetime import date

today = date.today().isoformat()   # 예: '2026-03-06'

# 1) 제약: 키를 정하고 가장 먼저 건다. 지하철은 역 이름이 곧 그 역이라 name 이 키다
#    :Station 의 name 과 :Line 의 name 에 각각 건다. 제약이 인덱스를 겸한다
run_cypher("CREATE CONSTRAINT station_name IF NOT EXISTS "
           "FOR (s:Station) REQUIRE s.name IS NODE KEY")
run_cypher("CREATE CONSTRAINT line_name IF NOT EXISTS "
           "FOR (l:Line) REQUIRE l.name IS NODE KEY")

# 2) 노드: 호선 두 개와 역 열일곱 개를 넣는다. 관계는 아직 손대지 않는다
#    :Line    - 역수·총연장·기점·종점은 원본에서 온 값이라 일반 SET 으로 매번 덮어쓴다
#    :Station - 원본에서 온 값은 이름뿐이고 그것이 곧 키다.
#               남길 것은 적재 기록이라 ON CREATE / ON MATCH 로 갈라 쓴다
for line in (line1, line2):
    run_cypher("""
MERGE (l:Line {name: $name})
SET l.station_count = $station_count, l.length_km = $length_km,
    l.first_station = $first_station, l.last_station = $last_station
""", **line)
run_cypher("""
UNWIND $rows AS row
MERGE (s:Station {name: row.name})
ON CREATE SET s.created_at = date($today)
ON MATCH SET  s.updated_at = date($today)
""", rows=[{"name": n} for n in stations1 + stations2], today=today)

# 3) 관계: 양 끝이 다 있는 지금 잇는다. 노드와 마찬가지로 UNWIND 로 묶어 보낸다
#    :SERVES - 그 호선이 이 역에 선다. 운행 관계 열여덟 개.
#              stop_order(그 호선에서 몇 번째 역인지)·cumulative_km(기점부터 누계)
serves_rows = [{"line": ln, "station": st, "stop_order": q, "cumulative_km": km}
               for ln, rows in (("1", serves1), ("2", serves2)) for st, q, km in rows]
run_cypher("""
UNWIND $rows AS row
MATCH (l:Line {name: row.line}), (s:Station {name: row.station})
MERGE (l)-[r:SERVES]->(s)
SET r.stop_order = row.stop_order, r.cumulative_km = row.cumulative_km
""", rows=serves_rows)

#    :NEXT - 이 역 다음이 그 역이다. 인접 관계 열여섯 개.
#            distance_km·travel_time 은 구간의 값이라 노드가 아니라 관계에 붙는다.
#            중괄호의 line 이 이 관계를 가르는 키다(같은 두 역이 여러 호선으로 이어질 수 있다)
next_rows = [{"line": ln, "src": a, "dst": b, "distance_km": km, "travel_time": t}
             for ln, rows in (("1", next1), ("2", next2)) for a, b, km, t in rows]
run_cypher("""
UNWIND $rows AS row
MATCH (a:Station {name: row.src}), (b:Station {name: row.dst})
MERGE (a)-[r:NEXT {line: row.line}]->(b)
SET r.distance_km = row.distance_km, r.travel_time = row.travel_time
""", rows=next_rows)

# 4) 다 이었으면 세어서 채운다: 역마다 붙은 호선 수를 세어 line_count 와 transfer 를 덮어쓴다
#    누적이 아니라 매번 새로 세기 때문에 몇 번을 다시 돌려도 값이 같다
run_cypher("""
MATCH (l:Line)-[:SERVES]->(s:Station)
WITH s, count(l) AS line_count
SET s.line_count = line_count, s.transfer = line_count >= 2
""")

print(run_cypher("""
MATCH (s:Station) WITH count(s) AS 역
MATCH (l:Line) WITH 역, count(l) AS 호선
MATCH ()-[r:SERVES]->() WITH 역, 호선, count(r) AS 운행
MATCH ()-[r:NEXT]->() RETURN 역, 호선, 운행, count(r) AS 인접
""")[0])

### 의료 지식그래프 적재

같은 네 단계인데 **키가 `id`** 라는 점만 다릅니다.

In [ ]:
# 1) 제약: 이 자료는 id 가 키다. 약물과 약효분류가 같은 이름을 쓸 수 있어 이름으로는 못 가린다
#    :Compound 의 id 와 :Disease 의 id 에 각각 건다
run_cypher("CREATE CONSTRAINT compound_id IF NOT EXISTS "
           "FOR (c:Compound) REQUIRE c.id IS NODE KEY")
run_cypher("CREATE CONSTRAINT disease_id IF NOT EXISTS "
           "FOR (d:Disease) REQUIRE d.id IS NODE KEY")

# 2) 노드: 약물 여섯 개와 질병 세 개. 레이블만 다르고 넣는 모양이 같아 f-string 으로 돌린다
#    키는 id, 이름은 원본에서 온 값이라 일반 SET, 적재 기록은 ON CREATE / ON MATCH 로 갈라 쓴다
for label, rows in (("Compound", compounds), ("Disease", diseases)):
    run_cypher(f"""
UNWIND $rows AS row
MERGE (n:{label} {{id: row.id}})
ON CREATE SET n.created_at = date($today)
ON MATCH SET  n.updated_at = date($today)
SET n.name = row.name
""", rows=rows, today=today)

# 3) 관계: (:Compound)-[:TREATS]->(:Disease) 여덟 개. 양 끝을 id 로 찾아 잇는다
#    이 관계에는 붙일 속성이 없어 MERGE 만 하고 SET 은 없다
run_cypher("""
UNWIND $rows AS row
MATCH (c:Compound {id: row.src}), (d:Disease {id: row.dst})
MERGE (c)-[:TREATS]->(d)
""", rows=[{"src": a, "dst": b} for a, b in treats])

print(run_cypher("""
MATCH (c:Compound) WITH count(c) AS 약물
MATCH (d:Disease) WITH 약물, count(d) AS 질병
MATCH ()-[r:TREATS]->() RETURN 약물, 질병, count(r) AS 치료
""")[0])

In [ ]:
# 두 스크립트를 다 돌린 뒤 전체 크기를 다시 잰다
print(run_cypher("""
MATCH (n) WITH count(n) AS 노드
MATCH ()-[r]->() RETURN 노드, count(r) AS 관계
""")[0])

> 노드도 관계도 늘지 않았습니다. 전부 `MERGE` 로 넣었고 키가 정해져 있어서, 두 번째 실행은 **있는 것을 찾기만** 합니다. 적재 코드가 갖춰야 할 성질이 이것입니다.

---
## 이번 강의 정리

| 절 | 문법 | 핵심 |
|---|---|---|
| 1 | `CREATE CONSTRAINT … REQUIRE s.name IS NODE KEY` | 키를 정하고 **적재보다 먼저** 건다 |
| 1 | 키는 현실에서 고른다 | 파일에서 안 겹친다고 키가 되는 것이 아니다 |
| 1 | 중복이 있으면 제약 생성이 실패한다 | 나중에 걸려면 중복부터 정리해야 한다 |
| 1 | `IS UNIQUE` 는 **빈 값을 막지 않는다** | 그래서 키에는 `IS NODE KEY` 를 쓴다 |
| 2 | `MERGE (n:L {key}) SET n.x = …` | `MERGE` 에는 **식별 키만**, 바뀔 값은 `SET` |
| 2 | `ON CREATE SET` / `ON MATCH SET` | **적재가 남기는 기록**에 쓴다. 원본에서 온 값은 `SET` |
| 2 | `UNWIND $rows AS row MERGE …` | 여러 줄을 **한 번에** 보낸다. 순서 규칙은 그대로 |
| 3 | `MATCH (a),(b) MERGE (a)-[:R]->(b)` | 양 끝을 **같은 질의 안에서** 찾아 잇는다 |
| 3 | `MERGE (a)-[r:NEXT]->(b) SET r.x = …` | 두 노드 **사이의 값**은 관계에 매단다 |
| 3 | `WITH s, count(l) AS n SET s.x = n` | 누적하지 말고 **매번 세어 덮어쓴다**(멱등) |

- 순서는 한 줄로 외웁니다. **제약을 먼저, 노드를 그다음, 관계는 마지막.**
- 제약을 먼저 거는 이유는 그때가 **가장 싸기** 때문입니다. 빈 그래프에는 한 줄이면 걸리지만, 데이터가 들어간 뒤에는 중복을 찾아 정리하는 일이 앞에 붙습니다.
- 관계를 마지막에 두는 이유는 **양 끝이 있어야 화살표를 그을 수 있기** 때문입니다.
- `MERGE` 에는 **식별 키만**, 바뀔 수 있는 값은 `SET` 으로. 이 둘을 지켜야 재실행이 안전합니다.
- 값이 **노드에 붙는지 관계에 붙는지**를 매번 판단하세요. 두 노드 사이의 값은 관계의 몫입니다.
- **원본에서 온 값**은 `SET` 으로 덮어쓰고, **적재가 남기는 기록**은 `ON CREATE`·`ON MATCH` 로 갈라 씁니다. 둘 다 속성이지만 값의 출처가 다릅니다.
- 넣은 뒤에는 **세어서** 확인합니다. 레이블·관계 타입을 콕 집어 세면 다른 조각과 섞이지 않습니다.
- **누적하는 값은 멱등하지 않습니다.** 세어서 넣을 값은 더하지 말고 매번 새로 세어 덮어씁니다.

## ⏭️ 예고: 다음 시간

오늘은 넣는 **차례**를 밟았습니다. 그런데 무엇을 노드로 두고 무엇을 관계로 이을지는 이미 정해진 것으로 놓고 시작했습니다. **다음 교안**에서는 그 판단 자체를 다룹니다. 자료는 새것으로 바뀌지만 묻는 것은 같습니다. 이 값을 속성에 둘지 노드로 뺄지, 분류를 레이블로 둘지 속성으로 둘지를 정하는 **스키마 설계**입니다.

그다음 교안에서는 오늘 조각을 떼어 온 **그 원본 파일**을 **`LOAD CSV`** 로 곧장 읽어 들이고, 큰 적재를 여러 번에 끊어 돌리는 법을 배웁니다. 국내 공공데이터의 `CP949` 인코딩도 그때 다룹니다.

도구는 바뀌지만 **오늘의 순서는 그대로**입니다. 제약을 먼저 걸고, 노드를 다 넣고, 관계를 잇습니다. 규모가 커질수록 이 순서를 어겼을 때의 대가가 커집니다.

수고하셨습니다!